In [ ]:
import ast

import numpy as np
import pandas as pd

In [ ]:
movies_df = pd.read_csv('dataset/tmdb_5000_movies.csv')
credits = pd.read_csv('dataset/tmdb_5000_credits.csv')

In [ ]:
movies_df.shape


In [ ]:
movies = movies_df.merge(credits, on='title')

In [ ]:
movies.head(1)

In [ ]:
# kya rakh na hai-
#genres,id,keywords,title,overview,cast,crew

movies=movies[['genres',"movie_id","keywords","title","overview","cast","crew"]]


In [ ]:
movies.head()


In [ ]:
movies.duplicated().sum()

In [ ]:
movies.isnull().sum()

In [ ]:
movies.dropna(inplace=True)

In [ ]:
movies.iloc[0].genres

In [ ]:
#[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

#upare wale ko iss form mai lana hai= ["Action","Adventure","Fantasy","Science Fiction"]

In [ ]:
import ast
def convert(obj):
    L=[]
    for i in ast.literal_eval(obj):
        L.append(i["name"])
    return L




In [ ]:
import ast
def convert3(obj):
    L=[]
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i["name"])
            counter += 1

        else:
            break

    return L


In [ ]:
def fetch_director(obj):
    L=[]
    for i in ast.literal_eval(obj):
        if i["job"]== "Director":
            L.append(i["name"])
            break
    return L



In [ ]:
movies["genres"] = movies["genres"].apply(convert)
movies["keywords"] = movies["keywords"].apply(convert)
movies["cast"]=movies["cast"].apply(convert3)
movies["crew"]=movies["crew"].apply(fetch_director)

In [ ]:
movies["overview"]=movies["overview"].apply(lambda x:x.split()  )

In [ ]:
#removing spaces between words , like science fition= sciencefiction

movies["crew"]=movies["crew"].apply(lambda x:[i.replace(" ","") for i in x] )
movies["cast"]=movies["cast"].apply(lambda x:[i.replace(" ","") for i in x] )
movies["genres"]=movies["genres"].apply(lambda x:[i.replace(" ","") for i in x] )
movies["keywords"]=movies["keywords"].apply(lambda x:[i.replace(" ","") for i in x] )

In [ ]:
movies["tags"]=movies["overview"]+movies["genres"]+movies["keywords"]+movies["crew"]+movies["cast"]

In [ ]:
movies.head()

In [ ]:
new_df = movies[["movie_id", "title", "tags"]].copy()


In [ ]:
new_df["tags"] = new_df["tags"].apply(lambda x: " ".join(x) if isinstance(x, list) else x)
new_df.head()

In [ ]:
new_df["tags"]=  new_df["tags"].apply(lambda x:x.lower())

In [ ]:
new_df.head()

In [ ]:
import nltk

In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [ ]:
def stem(text):
    y=[]
    for i in text.split():
       y.append(ps.stem(i))
    return " ".join(y)

In [ ]:
new_df["tags"]=new_df["tags"].apply(stem)

In [ ]:
new_df['tags'][0]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english')

In [ ]:
vectors=cv.fit_transform(new_df["tags"]).toarray()

In [ ]:
vectors[0]

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarity=cosine_similarity(vectors)

In [ ]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

In [ ]:
similarity[1]

In [ ]:
def recommend(movie):
    movie_index = new_df[new_df["title"] == movie].index[0]
    distances = similarity[movie_index]

    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [ ]:
recommend("Batman Begins")